# 2 — Domain-adversarial training (DANN)

Target domain `cartoon`, sources `photo`, `sketch`, `art_painting`.

A shared `ResNet18` feature extractor **Gf** feeds two heads: a 7-way label
predictor **Gy** trained on source labels, and a binary domain classifier
**Gd** trained to tell source from target. Between Gf and Gd sits a
*gradient reversal layer* — identity on the forward pass, multiplication by
`-lambda` on the backward pass. Minimising the domain loss therefore pushes
Gd to separate the domains while pushing Gf to make them inseparable, and
the fixed point is a feature space in which domain identity is not
recoverable but class identity still is.

```
                        +--> Gy --> class (7)     CE on source labels only
Gf (ResNet18, 512-d) ---+
                        +--> GRL(-lambda) --> Gd --> domain (2)   CE on all
```

Loss: `L = L_y(source) + L_d(source + target)`. Adam, lr 1e-4, batch 32 per
domain stream, four epochs. Target labels are used for the final evaluation
only.

Feature-space effect is read off a t-SNE of the 512-d features for the
pooled source and target sets, before and after training, together with the
distance between the two centroids.

## 2.1 Model, data, and the untrained measurement

Accuracy before any training, for reference.

In [ ]:
# Domain-adversarial network: a shared ResNet18 feature extractor, a 7-way
# label head, and a binary source/target domain head sitting behind a
# gradient reversal layer. `lambda_` scales the reversed gradient on its way
# back into the feature extractor; it defaults to 0.0 here, which is what we
# want for the untrained measurement at the bottom of the cell.
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import random
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, lambda_):
        ctx.lambda_ = lambda_
        return input.view_as(input)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
    def forward(self, x):
        x = self.features(x)
        return x.view(x.size(0), -1)
class LabelPredictor(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)
class DomainClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
    def forward(self, x, lambda_):
        x = GRL.apply(x, lambda_)
        return self.net(x)
class DANN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.feature = FeatureExtractor()
        self.label = LabelPredictor(num_classes)
        self.domain = DomainClassifier()
    def forward(self, x, lambda_=0.0):
        feat = self.feature(x)
        return self.label(feat), self.domain(feat, lambda_)
class PACSDatasetWithDomain(Dataset):
    def __init__(self, dataset, domain_label):
        self.dataset = dataset
        self.domain_label = domain_label
    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        return x, y, torch.tensor(self.domain_label)
    def __len__(self):
        return len(self.dataset)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
data_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['photo', 'cartoon', 'sketch', 'art_painting']
target_domain = 'cartoon'
source_domains = [d for d in domains if d != target_domain]
def get_loaders():
    source_datasets = []
    for domain in source_domains:
        ds = datasets.ImageFolder(os.path.join(data_root, domain), transform=transform)
        source_datasets.append(PACSDatasetWithDomain(ds, domain_label=0))
    source_dataset = torch.utils.data.ConcatDataset(source_datasets)
    source_loader = DataLoader(source_dataset, batch_size=32, shuffle=False)
    target_ds = datasets.ImageFolder(os.path.join(data_root, target_domain), transform=transform)
    target_loader = DataLoader(PACSDatasetWithDomain(target_ds, domain_label=1), batch_size=32, shuffle=False)
    return source_loader, target_loader
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            preds, _ = model(x)
            correct += (preds.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total
model = DANN().to(device)
source_loader, target_loader = get_loaders()
src_acc = evaluate(model, source_loader)
tgt_acc = evaluate(model, target_loader)
print(f"Initial Accuracy on Source Domains: {src_acc * 100:.2f}%")
print(f"Initial Accuracy on Target Domain ({target_domain}): {tgt_acc * 100:.2f}%")

## 2.2 t-SNE helpers

`plot_tsne_overlap_analysis` colours points by domain and prints the
distance between the source and target centroids in the embedding;
`plot_tsne_by_class` colours the same points by class.

The centroid distance is a coarse proxy — t-SNE distances are not metric
and depend on perplexity and initialisation, so the number is only
meaningful compared against itself across the two runs of the same
embedding configuration.

In [ ]:
from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
def plot_tsne_overlap_analysis(features, domain_labels, title="t-SNE Domain Overlap"):
    tsne = TSNE(n_components=2, init='pca', random_state=0)
    reduced = tsne.fit_transform(features)
    reduced = np.array(reduced)
    domain_labels = np.array(domain_labels)
    source_points = reduced[domain_labels == 0]
    target_points = reduced[domain_labels == 1]
    source_center = source_points.mean(axis=0)
    target_center = target_points.mean(axis=0)
    dist = np.linalg.norm(source_center - target_center)
    plt.figure(figsize=(10, 6))
    plt.scatter(source_points[:, 0], source_points[:, 1], alpha=0.3, label='Train (Source)', c='blue',s=10)
    plt.scatter(target_points[:, 0], target_points[:, 1], alpha=0.3, label='Test (Target)', c='red',s=10)
    plt.scatter(*source_center, c='blue', marker='X', s=10, edgecolors='black', label='Source Centroid')
    plt.scatter(*target_center, c='red', marker='X', s=10, edgecolors='black', label='Target Centroid')
    plt.title(f"{title} - Distance = {dist:.2f}")
    plt.legend()
    plt.grid(True)
    plt.show()
    print(f"Distance between source and target centroids: {dist:.4f}")

In [ ]:
def plot_tsne_by_class(features, labels, title="t-SNE by Class"):
    tsne = TSNE(n_components=2, init='pca', random_state=0)
    reduced = tsne.fit_transform(features)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap="tab10", alpha=0.7,s=10)
    plt.legend(*scatter.legend_elements(), title="Class")
    plt.title(title)
    plt.show()

## 2.3 Training

Source and target streams are drawn in parallel and concatenated, so each
step sees 32 source and 32 target images. The domain head is trained on all
64; the label head only on the source half.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import random
import itertools
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, lambda_):
        ctx.lambda_ = lambda_
        return input.view_as(input)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
    def forward(self, x):
        x = self.features(x)
        return x.view(x.size(0), -1)
class LabelPredictor(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)
class DomainClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )
    def forward(self, x, lambda_):
        x = GRL.apply(x, lambda_)
        return self.net(x)
class DANN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.feature = FeatureExtractor()
        self.label = LabelPredictor(num_classes)
        self.domain = DomainClassifier()
    def forward(self, x, lambda_=0.00006):
        feat = self.feature(x)
        return self.label(feat), self.domain(feat, lambda_)
class PACSDatasetWithDomain(Dataset):
    def __init__(self, dataset, domain_label):
        self.dataset = dataset
        self.domain_label = domain_label
    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        return x, y, torch.tensor(self.domain_label)
    def __len__(self):
        return len(self.dataset)
data_root = os.environ.get("PACS_ROOT", os.path.join("PACS", "kfold"))
domains = ['photo', 'cartoon', 'sketch', 'art_painting']
target_domain = 'cartoon'
source_domains = [d for d in domains if d != target_domain]
# Evaluation transform, no augmentation. The original applied the augmented
# `transform` below to the target domain as well and measured target accuracy
# through it, so the headline 71.67% is accuracy on colour-jittered, blurred,
# noised cartoon images. Both numbers are reported at the end of `run()`.
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.01):
        self.mean = mean
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    AddGaussianNoise(0., 0.01),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
def get_loaders():
    source_datasets = []
    for domain in source_domains:
        ds = datasets.ImageFolder(os.path.join(data_root, domain), transform=transform)
        source_datasets.append(PACSDatasetWithDomain(ds, domain_label=0))
    source_dataset = torch.utils.data.ConcatDataset(source_datasets)
    source_loader = DataLoader(source_dataset, batch_size=32, shuffle=True, num_workers=0)
    target_ds = datasets.ImageFolder(os.path.join(data_root, target_domain), transform=transform)
    target_loader = DataLoader(PACSDatasetWithDomain(target_ds, domain_label=1), batch_size=32, shuffle=True)
    return source_loader, target_loader
def train_epoch(model, batch, optimizer, loss_c, loss_d, lambda_, is_src_mask):
    model.train()
    x, y, d = [t.to(device) for t in batch]
    optimizer.zero_grad()
    y_pred, d_pred = model(x, lambda_)
    loss_class = loss_c(y_pred[is_src_mask], y[is_src_mask])
    loss_domain = loss_d(d_pred, d)
    loss = loss_class + loss_domain
    loss.backward()
    optimizer.step()
    # Accuracy over the source half only. The original averaged over the whole
    # concatenated batch, which uses the target domain's labels -- unavailable
    # under the domain-generalisation protocol.
    acc = (y_pred.argmax(1)[is_src_mask] == y[is_src_mask]).float().mean().item()
    return loss.item(), acc
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            y_pred, _ = model(x)
            correct += (y_pred.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total
def extract_features(model, loader):
    model.eval()
    feats, labels, domains = [], [], []
    with torch.no_grad():
        for x, y, d in loader:
            x = x.to(device)
            f = model.feature(x)
            feats.append(f.cpu())
            labels.append(y)
            domains.append(d)
    return torch.cat(feats), torch.cat(labels), torch.cat(domains)
def run():
    source_loader, target_loader = get_loaders()
    model = DANN(num_classes=7).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_class = nn.CrossEntropyLoss()
    loss_domain = nn.CrossEntropyLoss()
    # Gradient-reversal strength. At 6e-5 the reversed gradient reaching the
    # feature extractor is ~4 orders of magnitude smaller than the
    # classification gradient, so the adversarial term barely moves the
    # features: the domain head still learns (its own gradients are not
    # scaled), but it is not confusing anything. The gain reported below is
    # therefore mostly ordinary fine-tuning on three domains. The published
    # DANN schedule ramps lambda from 0 to ~1 over training; that is the knob
    # to turn first. Left at the original value so the reported number
    # reproduces.
    lambda_ = 0.00006
    combined_dataset = torch.utils.data.ConcatDataset([source_loader.dataset, target_loader.dataset])
    combined_loader = DataLoader(combined_dataset, batch_size=32, shuffle=False)
    f0, l0, d0 = extract_features(model, combined_loader)
    plot_tsne_overlap_analysis(f0.numpy(), d0.numpy(), title="Before Training")
    plot_tsne_by_class(f0.numpy(), l0.numpy(), title="Before Training - by Class")
    num_epochs = 4
    for epoch in range(num_epochs):
        source_iter = iter(source_loader)
        target_iter = iter(target_loader)
        max_steps = max(len(source_loader), len(target_loader))
        for _ in range(max_steps):
            try:
                x_s, y_s, d_s = next(source_iter)
            except StopIteration:
                source_iter = iter(source_loader)
                x_s, y_s, d_s = next(source_iter)
            try:
                x_t, y_t, d_t = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                x_t, y_t, d_t = next(target_iter)
            x = torch.cat([x_s, x_t], dim=0)
            y = torch.cat([y_s, y_t], dim=0)
            d = torch.cat([d_s, d_t], dim=0)
            is_src = torch.cat([torch.ones(len(x_s)).bool(),
                                torch.zeros(len(x_t)).bool()])
            loss, acc = train_epoch(model, (x, y, d), optimizer,
                                    loss_class, loss_domain, lambda_, is_src)
        print(f"Epoch {epoch+1}/{num_epochs} - Last Batch Loss: {loss:.4f}, Accuracy: {acc:.4f}")
    acc_target = evaluate(model, target_loader)
    print(f"Target Domain Accuracy (augmented transform, as reported): {acc_target:.4f}")
    clean_target_ds = datasets.ImageFolder(os.path.join(data_root, target_domain),
                                           transform=eval_transform)
    clean_target_loader = DataLoader(PACSDatasetWithDomain(clean_target_ds, domain_label=1),
                                     batch_size=32, shuffle=False)
    acc_target_clean = evaluate(model, clean_target_loader)
    print(f"Target Domain Accuracy (clean transform): {acc_target_clean:.4f}")
    f1, l1, d1 = extract_features(model, combined_loader)
    plot_tsne_overlap_analysis(f1.numpy(), d1.numpy(), title="After Training")
    plot_tsne_by_class(f1.numpy(), l1.numpy(), title="After Training - by Class")
    return model
trained_model = run()

## 2.4 What the run showed

| | source/target centroid distance |
|---|---|
| before training | 6.2817 |
| after 4 epochs | 4.5311 |

Target (`cartoon`) accuracy went from 17.11% at initialisation to 71.67%.

Two caveats carry into the report:

- **The comparison baseline matters.** 71.67% should be read against
  cartoon's own leave-one-domain-out baselines from notebook 01 — 52.47%
  without augmentation and 66.13% with it — not against art_painting's
  32.08%/45.02%. The honest gain over the augmented baseline is about 5.5
  points.
- **`lambda_` is 6e-5.** The adversarial gradient reaching the feature
  extractor is four orders of magnitude below the classification gradient,
  so most of that gain is plain fine-tuning on three domains plus
  augmentation, and the centroid movement is what fine-tuning does to the
  embedding rather than evidence of adversarial alignment. Raising
  `lambda_` onto the published schedule is the first thing to try.